In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
import pickle

PROJECT_ROOT = Path.cwd().parents[2]

data_path = PROJECT_ROOT / "data" / "large_clean.csv"

if not data_path.exists():
    matches = list(PROJECT_ROOT.rglob("large_clean.csv"))
    if not matches:
        raise FileNotFoundError(
            f"Could not find large_clean.csv under {PROJECT_ROOT}"
        )
    data_path = matches[0]

df = pd.read_csv(data_path)

In [2]:
df.head()

,Unnamed: 0,event_name,r_fighter,b_fighter,winner,weight_class,is_title_bout,gender,method,finish_round,...,weight_diff,reach_diff,SLpM_total_diff,SApM_total_diff,sig_str_acc_total_diff,td_acc_total_diff,str_def_total_diff,td_def_total_diff,sub_avg_diff,td_avg_diff
0,0,UFC Fight Night: Ribas vs. Namajunas,Amanda Ribas,Rose Namajunas,Blue,Women's Flyweight,0,Women,Decision - Unanimous,5,...,0.00,2.54,0.94,-0.11,-0.01,0.04,-0.02,0.26,0.2,0.69
1,1,UFC Fight Night: Ribas vs. Namajunas,Karl Williams,Justin Tafa,Red,Heavyweight,0,Men,Decision - Unanimous,3,...,-13.16,12.70,-1.22,-3.32,-0.02,0.50,0.13,0.50,0.2,4.75
2,2,UFC Fight Night: Ribas vs. Namajunas,Edmen Shahbazyan,AJ Dobson,Red,Middleweight,0,Men,KO/TKO,1,...,0.00,-2.54,-0.69,-1.22,0.06,-0.37,-0.01,-0.02,0.3,0.57
3,3,UFC Fight Night: Ribas vs. Namajunas,Payton Talbott,Cameron Saaiman,Red,Bantamweight,0,Men,KO/TKO,2,...,0.00,7.62,2.73,-0.60,0.08,-0.28,0.00,0.43,-0.2,-0.91
4,4,UFC Fight Night: Ribas vs. Namajunas,Billy Quarantillo,Youssef Zalal,Blue,Featherweight,0,Men,Submission,2,...,0.00,-5.08,4.48,3.84,0.07,-0.11,-0.22,0.01,-0.2,-1.04


In [3]:
numeric_features = []
categorical_features = []

for column in df.columns:
    if pd.api.types.is_numeric_dtype(df[column]):
        numeric_features.append(column)
    else:
        categorical_features.append(column)

In [4]:
numeric_features = [
    "wins_total_diff", "losses_total_diff", "age_diff", "height_diff",
    "weight_diff", "reach_diff", "SLpM_total_diff", "SApM_total_diff",
    "sig_str_acc_total_diff", "td_acc_total_diff", "str_def_total_diff",
    "td_def_total_diff", "sub_avg_diff", "td_avg_diff",
]
categorical_features = ["weight_class", "gender", "r_stance", "b_stance"]
features = numeric_features + categorical_features

data = df.loc[df["winner"].isin(["Red", "Blue"])].copy()
X = data[features]
y = data["winner"]

# Preserve event order; verify it is chronological and avoid splitting an event.
split_at = int(len(data) * 0.8)
X_train, X_test = X.iloc[:split_at], X.iloc[split_at:]
y_train, y_test = y.iloc[:split_at], y.iloc[split_at:]

numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("pca", PCA()),
])
categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("numeric", numeric, numeric_features),
    ("categorical", categorical, categorical_features),
])
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

# Compare no PCA with several variance-retention choices using training data only.
search = GridSearchCV(
    model,
    {
        "preprocess__numeric__pca": [
            "passthrough",
            PCA(n_components=0.80),
            PCA(n_components=0.90),
            PCA(n_components=0.95),
        ]
    },
    cv=TimeSeriesSplit(n_splits=5),
    scoring="balanced_accuracy",
)
search.fit(X_train, y_train)

predictions = search.predict(X_test)
print("Best setting:", search.best_params_)
print("Holdout balanced accuracy:", balanced_accuracy_score(y_test, predictions))

Best setting: {'preprocess__numeric__pca': 'passthrough'}
Holdout balanced accuracy: 0.7375420455961131


In [5]:
model.__dict__

{'steps': [('preprocess',
   ColumnTransformer(transformers=[('numeric',
                                    Pipeline(steps=[('imputer',
                                                     SimpleImputer(strategy='median')),
                                                    ('scaler', StandardScaler()),
                                                    ('pca', PCA())]),
                                    ['wins_total_diff', 'losses_total_diff',
                                     'age_diff', 'height_diff', 'weight_diff',
                                     'reach_diff', 'SLpM_total_diff',
                                     'SApM_total_diff', 'sig_str_acc_total_diff',
                                     'td_acc_total_diff', 'str_def_total_diff',
                                     'td_def_total_diff', 'sub_avg_diff',
                                     'td_avg_diff']),
                                   ('categorical',
                                    Pipeline(steps=[('im

In [6]:
best_model = search.best_estimator_
predictions = best_model.predict(X_test)

print("Best CV balanced accuracy:", search.best_score_)
print("Holdout balanced accuracy:", balanced_accuracy_score(y_test, predictions))
print("\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))
print("Confusion matrix (rows=true, columns=predicted; Blue, Red):")
print(confusion_matrix(y_test, predictions, labels=["Blue", "Red"]))

Best CV balanced accuracy: 0.680799497158538
Holdout balanced accuracy: 0.7375420455961131

Classification report:
              precision    recall  f1-score   support

        Blue       0.13      0.83      0.23        92
         Red       0.98      0.65      0.78      1396

    accuracy                           0.66      1488
   macro avg       0.56      0.74      0.51      1488
weighted avg       0.93      0.66      0.75      1488

Confusion matrix (rows=true, columns=predicted; Blue, Red):
[[ 76  16]
 [490 906]]


In [7]:
model_dir = PROJECT_ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "model_log_reg.pkl"

with open(model_path, "wb") as file:
    pickle.dump(best_model, file)

print(f"Model saved to: {model_path}")

Model saved to: e:\Projects\UFC-Prediction-Model\models\model_log_reg.pkl
